# Linear regression for prediction

Signification des émoticones :
- 🌞 : documentations importantes
- 👀 : documentations intéressantes à connaître
- 🌚 : en complément
- (vide) : à vous de voir

-------

For the remainder of the module we will look at using linear regression in a more algorithmic/computer science/machine learning context. In this notebook we will take a more detailed look at conducting linear regression in `scikit-learn`, you will:
* write your own code to compute MSE
* Learn what training and testing sets are
* Use `scikit-learn` to perform a multiple linear regression
* Evaluate your results
* Create a model to predict housing prices


## Assesing model accuracy

We always must evaluate how 'good' our model is in order to understand how it performs. One common measure of accuracy (or error) for a regression type problem is **Mean Squared Error (MSE)**. This can be calculated using the following formula:

<div style="font-size: 120%;">  
$$ MSE = \frac{1}{n} \sum_{i=1}^n (y_i − \hat{f}(x_i))^2, $$
</div>

where $y_i$ is the true values and $\hat{f}(x_i)$ is the predicted values by our model over $n$ samples.

### Exercise 1: write a function to calculate MSE

Given the lists of true values and predictions write a function that calculates the MSE of this data. 


HINT
* using numpy arrays will make this much easier.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
true_values = [13.9, 20.3, 2.1, 8.8, 5.2]
predicted_vals = [14.3, 18.9, 3.4, 8.9, 5.0]

In [ ]:
### your solution here
def mse(true, predicted):
    if len(true) == len(predicted):
        n = len(true)
        add = 0
        for i in range(0,n):
            add = add + (true[i]-predicted[i])**2
        return add/n
    raise TypeError

In [ ]:
mse(true_values, predicted_vals)

### MSE evaluation in sklearn

`Scikit-Learn` provides a function to calculate MSE (as well as other useful performance metrics) so that you dont have to carry around your function to all the different projects you work on. We can use this to evaluate our function and check that we got the correct answer in the exercise above.

In [ ]:
from sklearn.metrics import mean_squared_error

mean_squared_error(true_values, predicted_vals)

**Task 1**: Other evaluation metrics
* Find at least 2 other metrics that exist in `Scikit-Learn` that can be used in regression problems.
* Calculate the errors of these on the hypothetical data we used above.

In [ ]:
#### your solution here
## Mean absolute error:Measures the average absolute difference between predicted and actual values.
#It is expressed in the same units as the target variable.
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(true_values, predicted_vals))

##Root mean squared error Measures the square root of the average squared prediction errors. 
#It penalizes large errors more strongly than MAE.
from sklearn.metrics import root_mean_squared_error
print(root_mean_squared_error(true_values, predicted_vals))

## Training and testing sets

When building models for prediction we are most interested in building a model that will generalise well to new sets of data that we will encounter in the future. Strictly speaking what we have calculated above is a *training MSE*, which means that the calculation of MSE was performed on the same data samples the model was fitted to. 
 
One good way to asses whether our model will be useful on new data sets is to fit our model only to a subset of the data, and then use new data the model has not seen to produce a prediction which we can evaluate. We saw this process in the previous notebook when we introduced `scikit-learn` and Logistic Regression.

Here is another example:

In [ ]:
from sklearn.datasets import load_diabetes
diabs = load_diabetes()

In [ ]:
print(diabs.DESCR)

In [ ]:
diabs.data.shape

In [ ]:
diabs.target.shape

In [ ]:
import pandas as pd

X = pd.DataFrame(diabs.data, columns=diabs.feature_names)
y = diabs.target

In [ ]:
X.head()

In [ ]:
y[:5]

We split the data into a test and training set.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 18)

Fitting a linear regression model.

In [ ]:
from sklearn.linear_model import LinearRegression

# fit model to training data
model = LinearRegression()
model.fit(X_train, y_train)

# predict y values for testing data set
y_pred_test = model.predict(X_test)

# evaluate MSE
mse_test = mean_squared_error(y_test, y_pred_test)
print('test MSE: {0}'.format(mse_test))

We can also use this model to predict values on the train set, and evaluate the MSE. Comparing the two MSE scores gives us information on whether the model is overfitting or not. Normally we would compare these scores with other types of models (for example *support vector machines* or *random forests*...) that we have built in order to understand which model is best for the particular problem we are trying to solve. We almost always expect the *training error* to be lower than the *test error* (i.e. our performance reduces, or MSE increases) slightly when we move to new data the model has not seen.

In [ ]:
y_pred_train = model.predict(X_train)
mse_train = mean_squared_error(y_train, y_pred_train)
print('training MSE: {0}'.format(mse_train))

## Note: interpreting MSE

MSE is not particularly intuitive to help understand how well your model predicts. In order to interpret how well our model is doing we could convert MSE to something with similar units to the target variable by taking the square root. This would be **Root Mean Squared Error** or **RMSE**.

<br/>

<div style="font-size: 120%;">
$$ RMSE = \sqrt{MSE}$$
</div>

In [ ]:
import numpy as np

In [ ]:
rmse_test = np.sqrt(mse_test)
print('RMSE: {0}'.format(rmse_test))

If we do some simple descriptive stats on the target variable for the training set, we can see that our model is finding a solution on average within one standard deviation of the variance of the target. This type of analysis can give us some confidence whether our model is remotely useful or not!

In [ ]:
mean  = np.mean(y_train)
std = np.std(y_train)
print('Mean (standard deviation) values of target variables in train set: {0} ({1})'.format(mean, std))

RMSE, MSE, R2 squared: further reading

* 🌚 https://stats.stackexchange.com/questions/242787/how-to-interpret-root-mean-squared-error-rmse-vs-standard-deviation
* 🌚 https://setosa.io/ev/ordinary-least-squares-regression/
* 👀 https://medium.com/analytics-vidhya/mae-mse-rmse-coefficient-of-determination-adjusted-r-squared-which-metric-is-better-cd0326a5697e

## Correlation between predictor variables (or features)

Correlation between two (or more) predictive variable means that there is a statistical relation between these two variables. This can be causal or not. When two variables are highle correlated (> 0.5), than they are collinear. This can be problematic.In the presence of collinearity between two or more predictors, it is difficult to isolate the impact of each of that predictor on the response. Multicollinearity has an impact on the coefficients and p values of those coefficients. But it does not impact the precision of the predictions. So depending on the goal (first notebook, classical statistics or machine learning) it is desirable to take out features that are correlated. Also in the case of ML, when we have a dataset with a large number of predictors, taking out variables that are collinear helps to reduce the dimensions and make a simpler, more stable model.


(Further reading, see page 73, 74, 99 of ILS book).
🌚 https://datascience.stackexchange.com/questions/24452/in-supervised-learning-why-is-it-bad-to-have-correlated-features

Two methods:
* make a correlation matrix and decide what variables to leave out. (0.1 = moderate, 0.5 = large)
* Use variance inflaction factor method to leave out columns / features (if VIF > 5, leave out column)

In [ ]:
# Correlation matrix
df = pd.concat([X, pd.Series(y)], axis=1)
df = df.rename(columns={0:'diabetes'})

import seaborn as sns
_ = sns.heatmap(df.corr(), cmap="RdBu_r", vmin=-1, vmax=1)

From the above matrix we see that bmi (predictor) is strongly correlated to diabetes (response). As a predictor, bmi is strongly related to bp. To make a regression model we could take out bp without impacting the accuracy of the prediction of diabetes. This results in a simpler and smaller model.

In [ ]:
# Variance Inflation Factor
from statsmodels.stats.outliers_influence import variance_inflation_factor

variables = X
variables.shape
vif = pd.DataFrame()
vif['VIF'] = [variance_inflation_factor(variables.values,i) for i in range(variables.shape[1])]
vif['features'] = variables.columns
vif[vif['VIF']>5]

Leaving out the above columns should not impact the precision of prediction. But it results in a smaller and simpler model. Also, the std error and confidence interval of the coefficients should be smaller. But be careful, deleting all features with high collinearity, may lead to deleting the feature that is the best predictor for the response. So first check the correlation between features you want to delete and the (cor)relation with the response variable.

## Exercise 2: Linear regression prediction of house prices

The California housing market is booming and, as it is the home of silicon valley, the housing company **Homes4You** have decided they want to use new data science techniques to aid their business. They want you to build a predictive model that provides the median house price for each local area across the state (These local areas are often called *'blocks'*). They will use this to help guide their agents in how to price houses.

They have gathered a data set with numerous features to assist in building your model. However, they prefer to include as few features as possible to minimize the time their agents will spend collecting data in the future.


Your mission is to apply a regression model to predict housing prices in California.
* Import the data the company has collected using the sklearn function `fetch_california_housing`
* Make a heatmap of the correlation coefficients between the variables
* Split the data into a training and testing set
* Build a linear regression model using all the variables available and print the train and test MSE scores.
* Try building other models with different combinations of variables. Print the train and test MSE scores of at least 2 of these models.
* Provide a recommendation to Homes4You of which variables you would include in the model.
* Print the train and test **RMSE** scores of your final model and explain in a few sentences to Homes4You how well your model is performing. 

In [ ]:
#### your solution here
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target
df = pd.concat([X, pd.Series(y)], axis=1)
df = df.rename(columns={0:'housePrice'})
_ = sns.heatmap(df.corr(), cmap="RdBu_r", vmin=-1, vmax=1)

In [ ]:
df.corr()['housePrice'].sort_values(ascending=False)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
def evaluate_model(features):
    X_train_selected = X_train[features]
    X_test_selected = X_test[features]
    model = LinearRegression()
    model.fit(X_train_selected,y_train)
    y_pred_test = model.predict(X_test_selected)
    mse_test = mean_squared_error(y_test, y_pred_test)
    print('test MSE : {0}'.format(mse_test))
    y_pred_train = model.predict(X_train_selected)
    mse_train = mean_squared_error(y_train, y_pred_train)
    print('training MSE : {0}'.format(mse_train))
    return mse_test, mse_train

todo_test,todo_train = evaluate_model(X.columns)   
ejercicio_test,ejercicio_train = evaluate_model(X.drop(columns = to_drop).columns)   
yo_test,yo_train = evaluate_model(X.drop(columns = ['HouseAge','AveOccup','Population','AveBedrms']).columns)   
min_test, min_train = evaluate_model(['MedInc','AveRooms']) 

otro_test,otro_train = evaluate_model(X.drop(columns = ['Population']).columns)  
otro2_test,otro2_train = evaluate_model(X.drop(columns = ['AveBedrms', 'Population', 'AveOccup']).columns)   

In [ ]:
X_train

In [ ]:
ejercicio_test_rmse, ejercicio_train_rmse = np.sqrt(evaluate_model(X.drop(columns = to_drop).columns))
print('test rmse: '+str(ejercicio_test_rmse))
print('train rmse: '+str(ejercicio_train_rmse))

MY ANSWER: I have chosen the model with the columns `['MedInc', 'HouseAge', 'Latitude', 'Longitude']` because it has one of the lowest test MSE (including 'AveRooms' makes it slightly better but I prefer to include one feauture less). Moreover, using only four features, its performance is very similar to that of the model using all eight variables.

This means that we can obtain almost the same predictive performance with fewer variables, reducing the amount of information that Homes4You agents need to collect.

The final model has a training RMSE of 0.734 and a testing RMSE of 0.741. These values are very similar, suggesting that the model generalizes reasonably well to unseen data.

Since the target is expressed in hundreds of thousands of dollars, the test RMSE corresponds to approximately $74,100.

If removing strongly collinear features doesn’t lead to a better model, you can also try different feature combinations (for example, the one below yields similar training and testing scores).

In [ ]:
# Leave out these columns and then compute the training and testing MSE
to_drop = ['AveRooms', 'AveBedrms', 'Population', 'AveOccup']
df_drop = df.drop(columns = to_drop)
df_drop.columns

**Task 3: Explain this to your neighbour...**
* The training and test errors are quite close. What does this mean?


The training error measures how well the model works on the data it has used to learn and the test error measures how well it works for data it has never seen. If they are similar, this means that i works similarly for known and uknown data.


<details><summary>HINT</summary><br>
HINT: think in terms of if we were to apply this fitted model to new data the model has not seen.

</details>

* If your MSE & RMSE results are different to those of your peers (even when using the same features and model), why might this be?

In the train-slip if each person uses a different division of the dataset, they will have diferent groups of training and test



**Task 4:**
* Try to load the 'boston house price' (do some further research, what is the issue here?).
  
It has been removed from scikit_Learn for ethical reasons
* Try to load 'Linnerrud' data set that comes with sklearn? Try to build a model

Multi-output regression: it predicts simoultaneusly 3 output variables with several input variables

In [ ]:
from sklearn.datasets import load_linnerud

linnerud = load_linnerud()
print(linnerud.data.shape)
print(linnerud.target.shape)

In [ ]:
print(linnerud.feature_names)
print(linnerud.target_names)

In [ ]:
X = pd.DataFrame(linnerud.data, columns=linnerud.feature_names)
y = linnerud.target
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
predictions.shape



## To summarize

* Several metrics can be used to measure the accuracy of predictions, such as:
> - Mean Squared Error (MSE), which penalizes larger errors more heavily.
> - Root Mean Squared Error (RMSE) : the square root of MSE, providing results in the original units of Y.
> - Mean Absolute Error (MAE), which gives the average magnitude of errors without squaring.

* The data is typically split into:
> - A training set (usually 75% or 80% of the data), used to fit the model.
> - A testing set (the remaining 20% or 25%), consisting of unseen data to evaluate the model’s ability to generalize to new observations.

* Comparing training MSE (low) and test MSE (higher) helps detect overfitting. A small gap indicates good generalization.

* Tools like correlation matrices and Variance Inflation Factor (VIF > 5) identify redundant features, enabling simpler, more stable models.

